In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv openAI langchain_core langchain_openai

In [3]:
# 환경변수 설정

In [4]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [5]:
# .env 파일에서 환경 변수 로드
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = OpenAI()

In [6]:
# <호출(invoke) 출력>

# 라이브러리 불러오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 오픈AI의 대규모 언어 모델 설정
model = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 주어진 주제에 대한 설명 요청
prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧게 설명을 해주세요")
# 출력 파서 정의: AI 메시지의 출력 내용을 추출
parser = StrOutputParser()
# 프롬프트, 모델, 출력 파서를 체인으로 연결
chain = prompt | model | parser

# 응답을 호출
chain.invoke({"topic":"더블딥"})


"더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체가 한 번 발생한 후 잠시 회복되었다가 다시 경기 침체에 빠지는 현상을 의미합니다. 이 과정은 일반적으로 두 번의 침체가 이어지는 'V자' 형태의 회복곡선으로 나타나며, 첫 번째 침체 이후 회복세가 약해지거나 외부 요인으로 인해 다시 하락하게 됩니다. 더블딥은 경제 전반에 부정적인 영향을 미치며, 고용, 소비, 투자 등 여러 경제 지표의 회복을 어렵게 만듭니다."

In [7]:
# <배치(Batch) 출력>

# 주어진 주제 리스트를 배치로 출력
chain.batch([{"topic": "더블딥"}, {"topic": "인플레이션"}])

['더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체가 한 번 발생한 후 잠시 회복되는 듯하다가 다시 한 번 침체에 빠지는 현상을 의미합니다. 즉, 경제가 일시적으로 개선되는 것처럼 보이다가 다시 하락하는 형태로, 두 번의 경기 침체가 연속적으로 발생하는 상황을 말합니다. 이러한 현상은 기업 투자, 소비자 신뢰, 고용 시장 등 여러 가지 경제 지표에 영향을 미칠 수 있으며, 경제 정책 결정에 중요한 고려 요소가 됩니다.',
 '인플레이션은 일반적으로 물가가 지속적으로 상승하는 현상을 의미합니다. 이는 화폐의 구매력이 감소하게 만들어, 같은 금액으로 살 수 있는 상품이나 서비스의 양이 줄어드는 결과를 초래합니다. 인플레이션은 여러 요인에 의해 발생할 수 있으며, 대표적으로 수요 증가, 생산 비용 상승, 통화 공급의 확대 등이 있습니다. 적정 수준의 인플레이션은 경제 성장의 신호로 간주되지만, 지나치게 높은 인플레이션은 경제에 부정적인 영향을 미칠 수 있습니다.']

In [8]:
# <스트림(Stream) 출력>

# 응답을 토큰 단위로 스트리밍하여 출력
for token in chain.stream({"topic":"더블딥"}):
    # 스트리밍된 내용을 출력, 각 내용을 붙여서 출력하며, 버퍼를 즉시 플러시하여 실시간으로 보여줌
    print(token, end="", flush=True)


더블딥(Double Dip)은 경제학 및 금융에서 사용되는 용어로, 경기 침체가 두 번 발생하는 상황을 의미합니다. 첫 번째 경기 침체 후 경제가 일시적으로 회복되었다가 다시 두 번째 침체로 이어지는 경우를 가리킵니다. 이 현상은 경제가 완전히 회복되지 않고 불안정한 상태가 지속될 때 발생할 수 있으며, 투자자와 정책 입안자들에게 큰 도전이 됩니다. 더블딥은 일반적으로 실업률 증가, 소비 감소, 기업 투자 축소 등의 부정적인 경제 지표와 관련이 있습니다.

In [9]:
# <구성된 체인을 다른 러너블과 결합하기>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# "이 대답을 영어로 번역해 주세요"라는 질문을 생성하는 프롬프트 템플릿을 정의
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해 주세요: {answer}")

# 이전에 정의된 체인(chain)을 사용하여 대답을 생성하고, 그 대답을 영어로 번역하도록 프롬프트에 전달한 후, 모델을 통해 결과를 생성하여 최종적으로 문자열로 파싱하는 체인을 구성
composed_chain = {"answer": chain} | analysis_prompt | model | StrOutputParser()
# "더블딥"이라는 주제로 대답을 생성하고, 체인 실행
composed_chain.invoke({"topic": "더블딥"})

'The term "Double Dip" is used in economics to refer to a situation where a recession occurs twice. It generally refers to cases where the first recession seems to end and the economy appears to be recovering, only to fall back into another recession. This phenomenon can occur when the economic recovery is incomplete or when additional shocks arise due to external factors. A double dip presents a significant challenge for economic policymakers and can lead to ongoing economic downturns and employment issues.'

In [10]:
# <람다 함수를 사용한 체인을 통해 구성하기>

# 이전에 정의된 값들
model = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("{topic}에 대해 짧게 설명을 해주세요.")
chain = prompt | model | StrOutputParser()
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해 주세요: {answer}")

# 람다 함수를 사용한 체인 구성
composed_chain_with_lambda = (
    # 이전에 정의된 체인(chain)을 사용하여 입력된 데이터를 받아옵니다.
    chain
    # 입력된 데이터를 "answer" 키로 변환하는 람다 함수를 적용합니다.
    | (lambda input: {"answer": input})
    # "answer " 키를 가진 데이터를 영어로 번역하도록 프롬프트에 전달합니다.
    | analysis_prompt
    # 프롬프트에서 생성된 요청을 모델에 전달하여 결과를 생성합니다.
    | model
    # 모델에서 반환된 결과를 문자열로 파싱합니다.
    | StrOutputParser()
)
# "더블딥"라는 주제로 대답을 생성하고, 그 대답을 영어로 번역합니다.
composed_chain_with_lambda.invoke({"topic": "더블딥"})


'The term "Double Dip" is used in economics to describe a phenomenon where an economy experiences a brief recovery followed by a return to a state of recession. In other words, it refers to a situation where the economy falls into a recession, experiences a temporary recovery, and then faces a second downturn. This phenomenon can impact various economic indicators such as consumer confidence, employment rates, and production, and it can have a negative effect on the overall economy.'

In [11]:
# <`.pipe()`를 통해 체인 구성하기>

# (방법1) 여러 작업을 순차적으로 .pipe를 통해 연결하여 체인 구성하기
composed_chain_with_pipe = (
  # 이전에 정의된 체인(chain)으로 입력된 데이터를 받아옴
  chain
  # 입력된 데이터를 "answer" 키로 변환하는 람다 함수 적용
  .pipe(lambda input: {"answer": input})
  # analysis_prompt를 체인에 연결하여 설명을 영어로 번역하는 작업 추가
  .pipe(analysis_prompt)
  # 모델을 사용해 응답 생성
  .pipe(model)
  # 생성된 응답을 문자열로 파싱
  .pipe(StrOutputParser())
)
# "더블딥"이라는 주제로 체인을 실행하여 답변 생성
composed_chain_with_pipe.invoke({"topic": "더블딥"})


'The term "Double Dip" is commonly used in economics and financial markets to refer to a phenomenon where an economic recovery is temporary and followed by another recession. In other words, it describes a situation where the economy appears to have recovered only to experience a second downturn. This can also occur in the stock market, and it is used to explain instances where investors see a recovery that does not sustain and the market declines once again.'

In [12]:
# (방법2) 좀 더 간단하게 연결하기
composed_chain_with_pipe = chain.pipe(lambda input:{"answer":input}, analysis_prompt, model, StrOutputParser())

# "더블딥"이라는 주제로 체인을 실행하여 답변 생성
composed_chain_with_pipe.invoke({"topic": "더블딥"})

'The term "Double Dip" is primarily used in economics and finance to refer to the phenomenon of a recession occurring twice. It describes a process where the economy recovers after a recession, only to enter another recession again. This can happen when the economic growth rate temporarily rebounds but then declines again due to structural issues or external shocks. Double Dips pose significant challenges for investors and policymakers, highlighting the uncertainty of economic recovery.'

In [13]:
# <`RunnableParallel`을 이용한 체인 구성>
from langchain_core.runnables import RunnableParallel
# OpenAI 모델 초기화
model = ChatOpenAI()
# 한국어 설명 생성 프롬프트 체인
kor_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧은 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 영어 설명 생성 프롬프트 체인
eng_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 영어로 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 병렬 실행을 위한 RunnableParallel 설정
parallel_chain = RunnableParallel(kor=kor_chain, eng=eng_chain)
# 주제에 대한 한국어와 영어 설명 생성
result = parallel_chain.invoke({"topic": "더블딥"})
# 결과 출력
print("한글 설명:", result['kor'])
print("영어 설명:", result['eng'])


한글 설명: 더블딥(Double Deep)은 딥러닝 모델 중 하나로, 일반적인 딥러닝 모델보다 더 깊은 층(layer)으로 구성되어 있는 모델을 의미합니다. 이를 통해 데이터의 복잡한 패턴을 더 잘 학습하고 표현할 수 있어 성능이 더욱 향상될 수 있습니다. 하지만 더블딥은 학습에 시간과 계산 리소스가 많이 소요되고, 과적합(overfitting)이 발생할 수 있는 단점이 있습니다.
영어 설명: Double deep learning (DoubleDIP) is a novel deep learning framework that aims to improve the interpretability and generalization of deep neural networks by learning disentangled representations. It achieves this by simultaneously training two networks, where one network is tasked with learning the target task and the other network learns to predict the latent variables that are aligned with the factors of variation in the data. This approach helps to improve the robustness and efficiency of deep learning models, making them more suitable for real-world applications.


In [14]:
# <`RunnableParallel`를 이용한 체인 구성하기>
from langchain_core.runnables import RunnableParallel
# OpenAI 모델 초기화
model = ChatOpenAI()
# 한국어 설명 생성 프롬프트 체인
kor_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 영어 설명 생성 프롬프트 체인
eng_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 영어로 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 병렬 실행을 위한 RunnableParallel 설정
parallel_chain = RunnableParallel(kor=kor_chain, eng=eng_chain)
# 주제에 대한 한국어와 영어 설명 생성
result = parallel_chain.invoke({"topic": "더블딥"})
# 결과 출력
print("한글 설명:", result['kor'])
print("영어 설명:", result['eng'])


한글 설명: 더블딥(Double deep)은 딥 러닝 모델 중 하나로, 보통 두 개의 딥 러닝 네트워크를 쌓아서 사용하는 방법을 말합니다. 이를 통해 더 복잡한 문제를 해결하고 더 나은 성능을 얻을 수 있습니다. 이러한 방법은 딥러닝 분야에서 널리 사용되고 있으며, 다양한 응용 분야에서 성공적으로 적용되고 있습니다.
영어 설명: Double Deep is a type of warehouse storage system where pallets are stored two deep in a row, allowing for higher storage density compared to traditional single deep racking systems. This system requires specialized forklifts with extendable forks to reach and retrieve pallets from the back of the row.


In [15]:
# 참고
class CustomLCEL:
    def __init__(self, value):
        self.value = value  # 객체 생성 시 값을 초기화합니다.
    def __or__(self, other):
        if callable(other):
            return CustomLCEL(other(self.value))  # other가 함수일 경우, 함수를 호출하고 그 결과를 새로운 객체로 반환합니다.
        else:
            raise ValueError("Right operand must be callable")  # other가 함수가 아니면 에러를 발생시킵니다.
    def result(self):
        return self.value  # 현재 값을 반환합니다.
# 문자열 끝에 느낌표를 추가하는 함수
def add_exclamation(s):
    return s + "!"
# 문자열을 뒤집는 함수
def reverse_string(s):
    return s[::-1]
# 파이프라인을 생성하여 순차적으로 문자열 변환 작업을 수행합니다.
custom_chain = (
    CustomLCEL("랭체인 공부하기")  # "랭체인 공부하기"로 초기화된 객체 생성
    | add_exclamation  # 느낌표 추가
    | reverse_string  # 문자열 뒤집기
)
# 최종 결과를 출력합니다.
result = custom_chain.result()
print(result)
# 출력: !기하부공 인체랭


!기하부공 인체랭
